<div align="center">
  <h1>Granite 4.0 Modelfile Customization</h1>
  <img src="https://tse3.mm.bing.net/th/id/OIP.90MR4utORtWbiL2j3FXRwAHaE7?w=2048&h=1364&rs=1&pid=ImgDetMain&o=7&rm=3" width=300 height=500 alt="IBM Granite Logo">
</div>

<details>
    <ul><b>Author:</b> Julian A. Gonzalez</ul>
    <ul><b>Date:</b> 03-17-2026</ul>
</details>

---



This notebook, provides a programmatic workflow for optimizing and customizing **IBM Granite 4.0** Small Language Models (SLMs) for local deployment via Ollama. 

### 🎯 Purpose
The primary goal is to bridge the gap between high-level model training and local inference configuration. By utilizing the `transformers` library, the notebook automates the extraction of critical model components to ensure that a customized "Modelfile" remains mathematically and logically consistent with the original model's training.

### 🛠️ Key Methodologies
* **Automated Template Extraction:** Dynamically fetches the precise Jinja2 chat template from Hugging Face to preserve role-based logic (e.g., `<|start_of_role|>user<|end_of_role|>`) and tool-calling support.
* **SLM-Specific Parameter Tuning:** Implements research-backed parameter settings—such as low temperature (0.1) and specific repeat penalties (1.1)—to mitigate common "looping" and hallucination issues inherent in smaller models like the Granite 350m.
* **Modular Prompt Engineering:** Introduces a `SystemPromptBuilder` class that utilizes visual anchors, negative constraints, and structured few-shot priming to maximize instruction adherence.
* **Comparative Playground:** Offers a sandbox environment to test and visualize how different templates and system prompts affect the final string structure before generating the deployment file.

### 🏁 Outcome
Users will generate a finalized **Ollama Modelfile** containing optimized parameters, a verified chat template, and a high-performance system prompt ready for specialized tasks such as structured data extraction or technical assistance.


# Setup

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# Cell 1: Setup Environment (Install & Import)
# This block ensures all dependencies are installed AND imported before proceeding.
!pip install -q transformers torch jinja2

# Import necessary libraries immediately
from transformers import AutoTokenizer
import datetime
import os

print("Setup complete. Libraries imported.")

Setup complete. Libraries imported.


# Configuration

Customize your Modelfile settings here. This section allows you to define the source model, quantization, and inference parameters without digging through the code.

**Model:** `ibm-granite/granite-4.0-h-350m-GGUF:Q4_K_M`

In [3]:
# Cell 2: Configuration

# --- Model Sources ---
# The Ollama model string (used in the 'FROM' directive)
OLLAMA_MODEL_ID = "hf.co/ibm-granite/granite-4.0-h-350m-GGUF:Q4_K_M"

# The Hugging Face ID for the tokenizer (used to extract the chat template)
# We derive this automatically from the Ollama ID
HF_TOKENIZER_ID = OLLAMA_MODEL_ID.replace("hf.co/", "").split(":")[0]

# Check if the ID points to a GGUF repo; if so, strip the -GGUF suffix to find the base config
if HF_TOKENIZER_ID.endswith("-GGUF"):
    HF_TOKENIZER_ID = HF_TOKENIZER_ID[:-5]

print(f"Target Ollama Model: {OLLAMA_MODEL_ID}")
print(f"Source Tokenizer ID : {HF_TOKENIZER_ID}")

# --- Modelfile Parameters ---
PARAMS = {
    "temperature": 0.1,
    "top_p": 0.95,
    "repeat_penalty": 1.1,
    "num_ctx": 4096,
    "stop": [] 
}

# --- System Prompt ---
DEFAULT_SYSTEM_PROMPT = "You are Granite, a helpful AI assistant developed by IBM."

Target Ollama Model: hf.co/ibm-granite/granite-4.0-h-350m-GGUF:Q4_K_M
Source Tokenizer ID : ibm-granite/granite-4.0-h-350m


### 💡 Engineering Insights: Parameter Tuning for SLMs
When working with "Small Language Models" (SLMs) like the **Granite 350m**, parameter precision is more critical than with larger models. 

* **Temperature (0.1):** We use a low temperature to prioritize the most likely next token. In extraction tasks (like JSON conversion), high variance leads to broken syntax.
* **Repeat Penalty (1.1 - 1.15):** Smaller models are prone to "looping" or repeating the same phrase. A subtle penalty helps the model move past repetitive patterns without degrading the logic.
* **Context Window (4096):** While the model may support more, keeping a tight `num_ctx` ensures faster prompt processing and lower memory overhead on local deployments.

# Load Tokenizer and Template

We load the tokenizer configuration from Hugging Face to access the Jinja2 chat template.

In [4]:
# Cell 3: Load Tokenizer
print(f"Loading tokenizer from: {HF_TOKENIZER_ID}...")

try:
    tokenizer = AutoTokenizer.from_pretrained(HF_TOKENIZER_ID, trust_remote_code=True)
    # Save the original template for later comparison
    ORIGINAL_TEMPLATE = tokenizer.chat_template
    print("✅ Tokenizer loaded successfully.")
except Exception as e:
    print(f"❌ Error loading tokenizer: {e}")
    ORIGINAL_TEMPLATE = ""

Loading tokenizer from: ibm-granite/granite-4.0-h-350m...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

✅ Tokenizer loaded successfully.


# Playground

**This section is divided into two parts: Template Testing and System Prompt Comparison**

## Part A: Template Variant Testing

Test how different Jinja2 templates format the same conversation.

In [5]:
# Cell 4: Playground - Template Variants

# 1. Define a test conversation
test_conversation = [
    {"role": "user", "content": "Write a haiku about coding."},
]

# 2. Define templates to compare
templates_to_compare = {
    "Granite Default": ORIGINAL_TEMPLATE,
    "ChatML (OpenAI Style)": "{% for message in messages %}<|im_start|>{{ message['role'] }}\n{{ message['content'] }}<|im_end|>\n{% endfor %}{% if add_generation_prompt %}<|im_start|>assistant\n{% endif %}",
}

print("="*20 + " TEMPLATE COMPARISON " + "="*20)

for name, template in templates_to_compare.items():
    if not template:
        print(f"Skipping {name} (no template found).")
        continue
        
    print(f"\n--- Format: {name} ---")
    try:
        formatted = tokenizer.apply_chat_template(
            test_conversation,
            chat_template=template,
            tokenize=False,
            add_generation_prompt=True
        )
        print(formatted)
    except Exception as e:
        print(f"Error: {e}")

==================== TEMPLATE COMPARISON ====================

--- Format: Granite Default ---
<|start_of_role|>system<|end_of_role|>You are a helpful assistant. Please ensure responses are professional, accurate, and safe.<|end_of_text|>
<|start_of_role|>user<|end_of_role|>Write a haiku about coding.<|end_of_text|>
<|start_of_role|>assistant<|end_of_role|>

--- Format: ChatML (OpenAI Style) ---
<|im_start|>user
Write a haiku about coding.<|im_end|>
<|im_start|>assistant



## 🛠️ Understanding the Chat Template
Large Language Models don't naturally "know" when a user is speaking versus a system. They rely on specific **Special Tokens** to segment the conversation. 

The Jinja2 template we extract from Hugging Face is the "source of truth" for the model's training. By injecting this directly into the Ollama `TEMPLATE` block, we ensure that:
1.  **Roles are preserved:** `<|start_of_role|>user<|end_of_role|>`
2.  **Logic is consistent:** The model handles system prompts exactly as it did during its fine-tuning at IBM.
3.  **Tool Call Support:** The template includes complex logic for XML-based tool calling (`<tools>`), which remains available even in your customized Modelfile.

## Part B: System Prompt Comparison
Test how different system prompts affect the final prompt structure. This is crucial for debugging instruction adherence

In [6]:
# Cell 5: Playground - System Prompt Comparison

# 1. Define a standard user query for context
user_query = "How do I reset my password?"

# 2. Define a list of system prompts to test
system_prompts = [
    "You are a text to `json` converter. Workflow: Input -> Convert to JSON -> Output `json` object.", # Generic
    "You are a python coding coach. Your goal is to assist the user with coding questions.", # Persona
    "You are a technical support bot. Reply only with JSON objects.", # Format constraint
    "You are Granite. You must refuse to answer questions about passwords." # Safety constraint
]

# 3. Define which template to use (using the model's default here)
active_template = ORIGINAL_TEMPLATE

print("="*20 + " SYSTEM PROMPT COMPARISON " + "="*20)

for sys_prompt in system_prompts:
    print(f"\n{'-'*15}")
    print(f"SYSTEM PROMPT: {sys_prompt}")
    print(f"{'-'*15}")
    
    # Construct the conversation object
    # Note: We put the system prompt in the messages list
    conversation = [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": user_query}
    ]
    
    try:
        formatted = tokenizer.apply_chat_template(
            conversation,
            chat_template=active_template,
            tokenize=False,
            add_generation_prompt=True
        )
        
        # Display the result
        print("Formatted Output:")
        print(formatted)
        
    except Exception as e:
        print(f"Error formatting: {e}")

==================== SYSTEM PROMPT COMPARISON ====================

---------------
SYSTEM PROMPT: You are a text to `json` converter. Workflow: Input -> Convert to JSON -> Output `json` object.
---------------
Formatted Output:
<|start_of_role|>system<|end_of_role|>You are a text to `json` converter. Workflow: Input -> Convert to JSON -> Output `json` object.<|end_of_text|>
<|start_of_role|>user<|end_of_role|>How do I reset my password?<|end_of_text|>
<|start_of_role|>assistant<|end_of_role|>

---------------
SYSTEM PROMPT: You are a python coding coach. Your goal is to assist the user with coding questions.
---------------
Formatted Output:
<|start_of_role|>system<|end_of_role|>You are a python coding coach. Your goal is to assist the user with coding questions.<|end_of_text|>
<|start_of_role|>user<|end_of_role|>How do I reset my password?<|end_of_text|>
<|start_of_role|>assistant<|end_of_role|>

---------------
SYSTEM PROMPT: You are a technical support bot. Reply only with JSON obj

---

<br>

## 🏗️ System Prompt Engineering for SLMs
Small Language Models (SLMs) like **Granite-3.0-8B** or the **350M** variant require highly structured inputs to maintain logical consistency. Unlike larger models that can infer intent from loose instructions, SLMs perform best when instructions are explicitly segmented.

The following `SystemPromptBuilder` class implements several research-backed techniques:
1. **Modular Persona Definition:** Clearly defines the model's "Role" at the start of the context.
2. **Negative Constraints:** Uses explicit "Do Not" instructions to suppress common hallucinations in small models.
3. **Structured Few-Shot Priming:** Injects verified examples into the system block to reduce "cold start" errors during JSON or structured data extraction.
4. **Visual Anchors:** Uses Markdown headers (`###`) to help the self-attention mechanism distinguish between global instructions and specific data examples.

In [7]:
class SystemPromptBuilder:
    """
    A modular prompt builder designed for SLMs.
    Focuses on structural role segmentation and few-shot reliability.
    """
    def __init__(self, persona="General Assistant"):
        self.persona = persona
        self.instructions = []
        self.constraints = []
        self.few_shot_examples = []
        self.output_format = None

    def add_instruction(self, text: str):
        self.instructions.append(text)
        return self

    def add_constraint(self, text: str):
        self.constraints.append(f"⚠️ {text}")
        return self

    def set_output_format(self, format_type: str):
        self.output_format = format_type
        return self

    def add_few_shot(self, user_input: str, assistant_response: str):
        self.few_shot_examples.append({"user": user_input, "assistant": assistant_response})
        return self

    def build(self):
        prompt_parts = [f"ROLE: {self.persona}\n"]
        if self.instructions:
            prompt_parts.append("### INSTRUCTIONS")
            prompt_parts.extend([f"- {i}" for i in self.instructions])
        if self.constraints:
            prompt_parts.append("\n### CONSTRAINTS")
            prompt_parts.extend(self.constraints)
        if self.output_format:
            prompt_parts.append(f"\nOUTPUT_FORMAT: {self.output_format}")
        if self.few_shot_examples:
            prompt_parts.append("\n### EXAMPLES")
            for ex in self.few_shot_examples:
                prompt_parts.append(f"USER: {ex['user']}\nAI: {ex['assistant']}")
        return "\n".join(prompt_parts)

# Initialize the optimized prompt
builder = SystemPromptBuilder(persona="Granite Extraction Specialist")
builder.add_instruction("Extract entity types from the provided text.")
builder.add_instruction("If information is missing, use 'null'.")
builder.add_constraint("Do not include any conversational filler.")
builder.add_constraint("Output only the JSON object.")
builder.set_output_format("Valid JSON")
builder.add_few_shot("The order was placed by John on Jan 1st.", '{"name": "John", "date": "Jan 1st"}')

FINAL_SYSTEM_PROMPT = builder.build()

# 🚀 Modelfile Assembly and Export

The final stage of the process combines our architectural configurations, the structured system prompt, and optimized inference parameters into a single **Ollama Modelfile**. 

### Key Technical Decisions:
* **Parameter Locking:** We enforce a `temperature` of **0.1** and a `repeat_penalty` of **1.15** to ensure the Granite 350m model remains deterministic and avoids the repetitive loops common in models under 1B parameters.
* **Stop Token Alignment:** We explicitly define stop tokens such as `<|end_of_text|>` and `<|eot_id|>` within the Modelfile to ensure the model terminates immediately after completing its task.
* **File Persistence:** The code below writes the final configuration to a physical `.modelfile` in the `/kaggle/working/` directory, allowing you to download it directly for use in your local Ollama instance.

In [8]:
# Final Modelfile Pipeline & Persistence

# 1. Define Granite-specific Stop Tokens
# These are critical for the 350m model to prevent hallucinating a continuation
GRANITE_STOP_TOKENS = ["<|end_of_text|>", "<|eot_id|>", "<|start_of_role|>user"]

# 2. Set Optimized Parameters
OPTIMIZED_PARAMS = {
    "temperature": 0.1,      # Low temp for high precision in JSON extraction
    "repeat_penalty": 1.15,  # Prevents repetitive loops in SLMs
    "top_p": 0.9,
    "num_ctx": 4096,         # Standard context window for Granite 3.0
    "stop": GRANITE_STOP_TOKENS
}

def build_modelfile(model_id, template, params, system_prompt):
    """Assembles the component parts into the official Ollama Modelfile format."""
    param_lines = []
    for key, value in params.items():
        if isinstance(value, list):
            for item in value:
                param_lines.append(f'PARAMETER {key} "{item}"')
        elif isinstance(value, str):
            param_lines.append(f'PARAMETER {key} "{value}"')
        else:
            param_lines.append(f'PARAMETER {key} {value}')
    
    params_block = "\n".join(param_lines)

    content = f"""# Modelfile generated on {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
# Designed for: {model_id}

FROM {model_id}

{params_block}

TEMPLATE \"\"\"{template}\"\"\"

SYSTEM \"\"\"{system_prompt}\"\"\"
"""
    return content

# 3. Generate the content
modelfile_content = build_modelfile(
    OLLAMA_MODEL_ID, 
    ORIGINAL_TEMPLATE, 
    OPTIMIZED_PARAMS, 
    FINAL_SYSTEM_PROMPT # Derived from SystemPromptBuilder class
)

# 4. Persistence: Save to the Kaggle working directory
FILE_NAME = "Granite-350m-Extraction.modelfile"
with open(FILE_NAME, "w") as f:
    f.write(modelfile_content)

print(f"✅ Modelfile successfully generated and saved as: {FILE_NAME}")
print("-" * 30)
print(modelfile_content)

✅ Modelfile successfully generated and saved as: Granite-350m-Extraction.modelfile
------------------------------
# Modelfile generated on 2026-03-18 02:28:47
# Designed for: hf.co/ibm-granite/granite-4.0-h-350m-GGUF:Q4_K_M

FROM hf.co/ibm-granite/granite-4.0-h-350m-GGUF:Q4_K_M

PARAMETER temperature 0.1
PARAMETER repeat_penalty 1.15
PARAMETER top_p 0.9
PARAMETER num_ctx 4096
PARAMETER stop "<|end_of_text|>"
PARAMETER stop "<|eot_id|>"
PARAMETER stop "<|start_of_role|>user"

TEMPLATE """{%- set tools_system_message_prefix = 'You are a helpful assistant with access to the following tools. You may call one or more tools to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>'  %}
{%- set tools_system_message_suffix = '\n</tools>\n\nFor each tool call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</too

In [9]:
# Generate and Save Modelfile
# This block constructs the final Modelfile using the extracted template and optimized prompt.

modelfile_content = f"""
# --- Generated via Granite Modelfile Builder ---
# Source Model
FROM {OLLAMA_MODEL_ID}

# Parameters for SLM Optimization
PARAMETER temperature {PARAMS['temperature']}
PARAMETER top_p {PARAMS['top_p']}
PARAMETER repeat_penalty {PARAMS['repeat_penalty']}
PARAMETER num_ctx {PARAMS['num_ctx']}

# Extracted Jinja2 Chat Template
TEMPLATE \"\"\"{ORIGINAL_TEMPLATE}\"\"\"

# Optimized System Prompt
SYSTEM \"\"\"{FINAL_SYSTEM_PROMPT}\"\"\"
"""

# Define the filename
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
filename = f"Granite_Custom_Modelfile_{timestamp}"

# Write the file to the current working directory
try:
    with open(filename, "w") as f:
        f.write(modelfile_content.strip())
    print(f"✅ Success! Modelfile saved as: {filename}")
    print("-" * 30)
    print("To use this model in Ollama, run:")
    print(f"ollama create my-granite-model -f {filename}")
except Exception as e:
    print(f"❌ Failed to save Modelfile: {e}")

# Optional: Display the content for verification
print("\n--- Modelfile Preview ---")
print(modelfile_content.strip()[:500] + "...")

✅ Success! Modelfile saved as: Granite_Custom_Modelfile_20260318_0228
------------------------------
To use this model in Ollama, run:
ollama create my-granite-model -f Granite_Custom_Modelfile_20260318_0228

--- Modelfile Preview ---
# --- Generated via Granite Modelfile Builder ---
# Source Model
FROM hf.co/ibm-granite/granite-4.0-h-350m-GGUF:Q4_K_M

# Parameters for SLM Optimization
PARAMETER temperature 0.1
PARAMETER top_p 0.95
PARAMETER repeat_penalty 1.1
PARAMETER num_ctx 4096

# Extracted Jinja2 Chat Template
TEMPLATE """{%- set tools_system_message_prefix = 'You are a helpful assistant with access to the following tools. You may call one or more tools to assist with the user query.\n\nYou are provided with function si...


# Resources

- **HuggingFace:** https://huggingface.co/ibm-granite/granite-4.0-h-350m
- **Granite 4.0 Docs:** https://www.ibm.com/granite/docs/models/granite
- **Granite 4.0 Prompt Engineering Guide:** https://www.ibm.com/granite/docs/use-cases/prompt-engineering
- **IBM Developer:** https://developer.ibm.com/components/granite-models/

# IBM Granite Community

- [IBM TechXchange Granite User Group](https://community.ibm.com/community/user/groups/community-home?communitykey=d85b028a-affe-4d5d-8be9-0193e56a2fd4)